# Experiment tracking: train PRD model, log to MLflow, register

This notebook:
1. Justifies the choice of `vol_regime + catboost` as the PRD model
2. Trains it via `ClassifierRunner`
3. Logs all params/metrics/artifacts to MLflow
4. Registers as `chronos_1h_prd` with `env=PRD` tag and `prd` alias

**Prerequisites:** run `mlflow_setup.ipynb` first to verify connectivity.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, str(pathlib.Path('.').resolve()))

from chronos_ts.tracking import configure_mlflow
configure_mlflow()

## Model selection justification

Based on the Phase 1 smoke test and EDA (README §7):

| Target | Test ROC-AUC (logreg) | Baseline balanced-acc |
|---|---|---|
| `vol_regime` | **0.691** | 0.333 |
| `large_move` | ~0.62 | 0.500 |
| `direction`  | 0.506 | 0.500 |

**Winner: `vol_regime`** — volatility is the one forecastable signal (HAR R²=0.085 from regression experiments). `catboost` is selected over `logreg` for nonlinear feature interactions.

Selection criteria: test edge corroborated on val (not a single-split fluke), trading metric coverage ≥ 10%, interpretability for the serving app.

In [ ]:
from scripts.train_classifier import ClassifierRunner, RunConfig

REGISTERED_MODEL = 'chronos_1h_prd'
EXPERIMENT      = 'chronos-1h-classification'

cfg = RunConfig(
    data_csv='outputs/datasets/btcusdt_clf_core.csv',
    label_family='vol_regime',
    model_name='catboost',
    output_dir='outputs/models/clf',
    cv_splits=3,
    seed=42,
)

result = ClassifierRunner(cfg).run(
    mlflow_experiment=EXPERIMENT,
    register_as=REGISTERED_MODEL,
    promote_to_prd=True,
)

In [ ]:
# Summary of key metrics
import pandas as pd

rows = []
for split in ('val', 'test'):
    m = result['metrics'][split]
    rows.append({
        'split': split,
        'balanced_acc': round(m['balanced_accuracy'], 4),
        'mcc': round(m['mcc'], 4),
        'roc_auc': round(m['roc_auc'], 4),
        'coverage': round(m['trading_coverage'], 3),
        'hit_rate': round(m.get('trading_hit_rate', float('nan')), 4),
    })
    bl = result['baselines'].get(split, {}).get('majority_class', {})
    rows.append({
        'split': f'{split}_baseline',
        'balanced_acc': round(bl.get('balanced_accuracy', float('nan')), 4),
        'mcc': round(bl.get('mcc', float('nan')), 4),
        'roc_auc': float('nan'),
        'coverage': 1.0,
        'hit_rate': float('nan'),
    })

pd.DataFrame(rows).set_index('split')

In [ ]:
# Confirm PRD alias
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()
aliases = client.get_model_version_by_alias(REGISTERED_MODEL, 'prd')
print(f'PRD model: {REGISTERED_MODEL} v{aliases.version}')
print(f'Tags: {aliases.tags}')
print(f'Run ID: {result.get("mlflow_run_id")}')

✅ Model logged and registered. Open `http://localhost:5050` to browse runs.

Next: `error_analysis.ipynb`